# 04 - Merge results (Light + Heavy) across RUN_IDs

**Mục tiêu:** gom `{RUN_ID}_metrics_light.json` và `{RUN_ID}_metrics_heavy.json` (từ notebook 03A/03B) thành:
- `{RUN_ID}_main_results_row_merged.csv`
- `{RUN_ID}_main_results_row_merged.json`

**Input**
- Một folder chứa các file JSON theo định dạng:
  - `{RUN_ID}_metrics_light.json`
  - `{RUN_ID}_metrics_heavy.json`

**Output**
- `results.csv`: bảng tổng hợp (mỗi RUN_ID 1 dòng)
- `results.json`: cùng nội dung ở dạng JSON list (phục vụ report / log)

**Ghi chú:**
- Nếu thiếu một trong hai file (light/heavy) của một RUN_ID, notebook vẫn tạo dòng cho RUN_ID đó và để trống các cột tương ứng.
- Thứ tự RUN_ID mặc định: `baseline` → `A` → `B` → `demo` → các RUN_ID khác theo alphabet (nếu có).


## 0) Config

In [ ]:
from pathlib import Path
import json
import re
import pandas as pd

# === USER CONFIG ===
# folder chứa {RUN_ID}_metrics_light.json & {RUN_ID}_metrics_heavy.json
METRICS_DIR = Path("/kaggle/input/vn-textbook-qwen2vl-03-metrics")   
OUT_DIR     = Path("/kaggle/working")   
OUT_PREFIX  = "results"                

## 1) Collect metrics files by RUN_ID

In [ ]:
PAT = re.compile(r"^(.+)_metrics_(light|heavy)\.json$")

by_run_id = {}  # run_id -> {"light": Path, "heavy": Path}

for p in METRICS_DIR.glob("*_metrics_*.json"):
    m = PAT.match(p.name)
    if not m:
        continue
    run_id, kind = m.group(1), m.group(2)
    by_run_id.setdefault(run_id, {})[kind] = p

# Fallback: hỗ trợ format cũ (không suffix) nếu không tìm thấy file *_<RUN_ID>.json
# Khi dùng fallback này, RUN_ID được đặt là "default".
if not by_run_id:
    light_legacy = METRICS_DIR / "metrics_light.json"
    heavy_legacy = METRICS_DIR / "metrics_heavy.json"
    if light_legacy.exists() or heavy_legacy.exists():
        by_run_id["default"] = {}
        if light_legacy.exists():
            by_run_id["default"]["light"] = light_legacy
        if heavy_legacy.exists():
            by_run_id["default"]["heavy"] = heavy_legacy

if not by_run_id:
    raise FileNotFoundError(
        f"Không tìm thấy metrics trong {METRICS_DIR}. "
        "Cần có {RUN_ID}_metrics_light.json và/hoặc {RUN_ID}_metrics_heavy.json."
    )

def run_id_key(run_id: str):
    # baseline trước, sau đó A/B, rồi phần còn lại
    if run_id == "baseline":
        return (0, run_id)
    if run_id in ("A", "B"):
        return (1, run_id)
    if run_id == "demo":
        return (2, run_id)
    return (3, run_id)

run_ids_sorted = sorted(by_run_id.keys(), key=run_id_key)
print("Found RUN_IDs:", run_ids_sorted)

## 2) Merge (light + heavy) per RUN_ID

In [ ]:
COLUMNS = ["Model", "Quote-CER", "Concept-Rec", "LLM-Score", "BERTScore", "BLEU-4", "METEOR"]

rows = []
merged_per_run_id = {}  # optional: keep merged dict per run_id

def _load_json(p: Path):
    return json.loads(p.read_text(encoding="utf-8"))

def _get(d: dict, k: str, default=""):
    v = d.get(k, default)
    return default if v is None else v

for run_id in run_ids_sorted:
    files = by_run_id[run_id]
    light = _load_json(files["light"]) if "light" in files else {}
    heavy = _load_json(files["heavy"]) if "heavy" in files else {}

    model_name = _get(light, "model_name", "") or _get(heavy, "model_name", "") or run_id

    row = {
        "Model": model_name,
        "Quote-CER": _get(light, "quote_cer", ""),
        "Concept-Rec": _get(light, "concept_rec", ""),
        "LLM-Score": _get(heavy, "llm_score_mean", ""),
        "BERTScore": _get(heavy, "bertscore_f1", ""),
        "BLEU-4": _get(light, "bleu4", ""),
        "METEOR": _get(light, "meteor", ""),
    }

    # Ép về float nếu có thể, giữ "" nếu trống
    for k in ["Quote-CER", "Concept-Rec", "LLM-Score", "BERTScore", "BLEU-4", "METEOR"]:
        if row[k] == "":
            continue
        try:
            row[k] = float(row[k])
        except Exception:
            pass

    rows.append(row)
    merged_per_run_id[run_id] = row

df = pd.DataFrame(rows, columns=COLUMNS)

## 3) Save merged outputs (all RUN_IDs)

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

csv_path  = OUT_DIR / f"{OUT_PREFIX}.csv"
json_path = OUT_DIR / f"{OUT_PREFIX}.json"

df.to_csv(csv_path, index=False)
json_path.write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")

print("Wrote:", csv_path)
print("Wrote:", json_path)

# (tuỳ chọn) Nếu bạn muốn debug theo RUN_ID:
(OUT_DIR / f"{OUT_PREFIX}_by_run_id.json").write_text(json.dumps(merged_per_run_id, ensure_ascii=False, indent=2), encoding="utf-8")

df